# MolmoAct2 x LIBERO — Long Suite Analysis

**Research question:** How does MolmoAct2 behave on the LIBERO Long suite (`libero_10`), and which task structures explain the remaining failures?

**Scope of this notebook:** the `libero_10` / Long suite only (10 tasks, 50 episodes each, 500 episodes total). This mirrors the per-suite notebook style used for `goal_suite_analysis.ipynb`, but adapts the analysis to the Long suite's artifacts and manual failure-video annotations.

Data comes from the final Long-suite evaluation outputs:
- `results.csv` — one row per episode
- `nlp_analysis_table.csv` or `scene_properties.csv` — one row per task, with scene/task properties when available
- `task_level_summary.csv` — per-task success and step statistics
- `failure_summary.csv` — failed episodes
- `all_failure_video_subgoal_annotations.csv` — manual failed-subgoal review

No extra GPU run and no extra model is needed below; everything is just re-slicing the existing evaluation outputs.

## Setup

Set `DATA_DIR` to wherever the `libero_10` output folder lives for this session.

- **Local / this repo:** this notebook lives in `docs/`, one level up from its data in `results/libero_10/`, so the default `../results/libero_10` works as-is.
- **Colab, cloned from GitHub:** clone the repo, then point `DATA_DIR` at `molmoact2-libero-eval/results/libero_10`.
- **Manual upload:** upload `results.csv`, `task_level_summary.csv`, `nlp_analysis_table.csv`, `failure_summary.csv`, and `all_failure_video_subgoal_annotations.csv` into one folder and set `DATA_DIR` to that folder.

The notebook is written to tolerate a few schema differences: for example, it can use either `success_rate` in `[0, 1]` form or `success_rate_percent` in percentage form.

In [ ]:
# !pip install -q plotly scipy kaleido opencv-python-headless

from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# Static PNG renderer helps GitHub display charts after the notebook is executed.
# If Kaleido is not installed locally, change this to "notebook_connected".
pio.renderers.default = "png"

DATA_DIR = Path("../results/libero_10")  # notebook lives in docs/, data lives in results/libero_10/

results = pd.read_csv(DATA_DIR / "results.csv")

def task_number(task_id):
    return int(str(task_id).split("_")[-1])

def normalize_percent(x):
    """Return success rate in percentage units regardless of whether source is 0-1 or 0-100."""
    x = float(x)
    return x * 100 if x <= 1.0 else x

# Load or build task-level table.
task_summary_path = DATA_DIR / "task_level_summary.csv"
if task_summary_path.exists():
    task = pd.read_csv(task_summary_path)
    task = task.rename(columns={
        "total_episodes": "n_episodes",
        "success_count": "n_success",
        "failure_count": "n_fail",
    })
    task["success_pct"] = task["success_rate"].apply(normalize_percent)
else:
    task = (
        results.groupby(["task_id", "instruction"], as_index=False)
        .agg(
            n_episodes=("success", "count"),
            n_success=("success", "sum"),
            avg_steps=("n_steps", "mean"),
            median_steps=("n_steps", "median"),
            min_steps=("n_steps", "min"),
            max_steps=("n_steps", "max"),
        )
    )
    task["n_fail"] = task["n_episodes"] - task["n_success"]
    task["success_pct"] = 100 * task["n_success"] / task["n_episodes"]

task["task_num"] = task["task_id"].apply(task_number)
task["task_short"] = "T" + task["task_num"].astype(str)
task = task.sort_values("task_num").reset_index(drop=True)

# Merge optional per-task scene/task metadata.
meta = None
for candidate in ["scene_properties.csv", "nlp_analysis_table.csv", "distractor_density.csv"]:
    p = DATA_DIR / candidate
    if p.exists():
        candidate_df = pd.read_csv(p)
        if "task_id" in candidate_df.columns:
            meta = candidate_df
            break

if meta is not None:
    keep_cols = [c for c in meta.columns if c not in task.columns or c in ["task_id"]]
    task = task.merge(meta[keep_cols], on="task_id", how="left", suffixes=("", "_meta"))

# Normalize common alternative column names.
if "success_rate_percent" in task.columns and "success_pct" not in task.columns:
    task["success_pct"] = task["success_rate_percent"].astype(float)

if "n_distractors" not in task.columns and "n_distractors_meta" in task.columns:
    task["n_distractors"] = task["n_distractors_meta"]
if "distractor_density" not in task.columns and "distractor_density_meta" in task.columns:
    task["distractor_density"] = task["distractor_density_meta"]

# Failure annotation file is optional but preferred for Long because exact failed subgoals
# were determined through manual video review rather than automatic subgoal logging.
ann_path = DATA_DIR / "all_failure_video_subgoal_annotations.csv"
annotations = pd.read_csv(ann_path) if ann_path.exists() else pd.DataFrame()

print(f"{len(results)} episodes across {task['task_id'].nunique()} tasks")
print(f"Loaded manual failure annotations: {len(annotations)} rows")
task[["task_id", "instruction", "success_pct", "n_episodes", "avg_steps", "n_fail"]]

## At a glance

Before detailed charts, this gives the headline numbers and a plain reference for what `T0`–`T9` mean.

In [ ]:
overall = 100 * results["success"].mean()
total_episodes = len(results)
total_success = int(results["success"].sum())
total_fails = total_episodes - total_success
n_perfect = int((task["success_pct"] == 100).sum())
worst = task.loc[task["success_pct"].idxmin()]

fig = go.Figure(go.Indicator(
    mode="number",
    value=overall,
    number={"suffix": "%", "font": {"size": 64, "color": "#2a78d6"}},
    title={"text": "Overall success rate<br><span style='font-size:0.6em;color:#898781'>"
                   f"{total_success} / {total_episodes} episodes, {len(task)} tasks, 50 each</span>"},
))
fig.update_layout(height=220, width=520, margin=dict(l=10, r=10, t=70, b=10))
fig.show()

print(f"Perfect tasks (100% success): {n_perfect} / {len(task)}")
print(f"Failed episodes: {total_fails} / {total_episodes}")
print(f"Weakest task: {worst['task_short']} ({worst['instruction']}) at {worst['success_pct']:.0f}%")
print()
print("Task reference (T0-T9 -> full instruction):")
for _, row in task.iterrows():
    print(f"{row['task_short']}: {row['instruction']}")

## 1. Success rate per task, with 95% confidence intervals

A task's success rate alone (for example, `92%`) does not say whether it is meaningfully different from another task's `100%`, especially at only 50 episodes per task. The Wilson interval gives each task's plausible range so we do not overclaim small gaps as real.

In [ ]:
def wilson_ci(successes, n, z=1.96):
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return center - half, center + half

ci_lo, ci_hi = [], []
for _, row in task.iterrows():
    n = int(row["n_episodes"])
    s = int(round(row["success_pct"] / 100 * n))
    lo, hi = wilson_ci(s, n)
    ci_lo.append(lo * 100)
    ci_hi.append(hi * 100)

task["ci_lo"], task["ci_hi"] = ci_lo, ci_hi

plot_df = task.sort_values("success_pct").reset_index(drop=True)
colors = ["#fab219" if row["ci_hi"] < 99.9 else "#2a78d6" for _, row in plot_df.iterrows()]

fig = go.Figure()
fig.add_bar(
    x=plot_df["task_short"],
    y=plot_df["success_pct"],
    marker_color=colors,
    error_y=dict(
        type="data",
        symmetric=False,
        array=plot_df["ci_hi"] - plot_df["success_pct"],
        arrayminus=plot_df["success_pct"] - plot_df["ci_lo"],
    ),
    customdata=plot_df[["instruction"]],
    showlegend=False,
    hovertemplate="%{x}: %{customdata[0]}<br>Success: %{y:.1f}%<extra></extra>",
)
fig.add_scatter(
    x=plot_df["task_short"],
    y=plot_df["ci_hi"] + 2,
    mode="text",
    text=plot_df["success_pct"].round(0).astype(int).astype(str) + "%",
    textfont=dict(size=11),
    showlegend=False,
    hoverinfo="skip",
)
fig.update_layout(
    title="libero_10 / Long: per-task success rate with 95% Wilson intervals",
    xaxis_title="Task",
    yaxis_title="Success rate (%)",
    yaxis_range=[75, 106],
    template="plotly_white",
    height=520,
    width=850,
    margin=dict(l=10, r=10, t=60, b=40),
)
fig.show()

task[["task_short", "instruction", "success_pct", "ci_lo", "ci_hi", "n_fail"]].sort_values("success_pct")

**Takeaway:** 6 of 10 Long tasks never fail. `T8` ("put both moka pots on the stove") is the clear outlier at 82%, and its confidence interval is separated from the perfect tasks. `T9`, `T6`, and `T3` are imperfect but still relatively high-success at 92%, 96%, and 98% respectively.

## 2. Failures per task

The confidence-interval chart shows uncertainty; this chart shows the raw number of failed episodes out of 50. This is the easiest way to see whether failures are spread across the suite or concentrated in a few tasks.

In [ ]:
fail_df = task[["task_id", "task_short", "instruction", "n_fail"]].copy().sort_values("n_fail")

fig = go.Figure()
fig.add_bar(
    x=fail_df["task_short"],
    y=fail_df["n_fail"],
    marker_color=["#e1e0d9" if n == 0 else "#fab219" for n in fail_df["n_fail"]],
    customdata=fail_df[["instruction"]],
    text=fail_df["n_fail"].astype(str),
    textposition="outside",
    hovertemplate="%{x}: %{customdata[0]}<br>Failures: %{y} / 50<extra></extra>",
)
fig.update_layout(
    title="libero_10 / Long: failed episodes per task (out of 50)",
    xaxis_title="Task",
    yaxis_title="Number of failures",
    yaxis_range=[0, max(10, fail_df["n_fail"].max() + 1)],
    template="plotly_white",
    height=500,
    width=850,
    margin=dict(l=10, r=10, t=60, b=40),
)
fig.show()

fail_df.sort_values("n_fail", ascending=False)

**Takeaway:** failures are task-clustered, not uniform. `T8` alone accounts for 9 of the 16 Long failures. `T9` accounts for 4, `T6` for 2, and `T3` for 1. The remaining 6 tasks have zero failures.

## 3. Average steps per task

Success rate says whether the model eventually completes the task. Step count says how much of the episode budget the policy uses while trying. For Long, the max step cap is 520, so a task with high average steps can be harder even if it often succeeds.

In [ ]:
steps_df = task.sort_values("avg_steps")

fig = go.Figure()
fig.add_bar(
    x=steps_df["task_short"],
    y=steps_df["avg_steps"],
    marker_color="#2a78d6",
    customdata=steps_df[["instruction"]],
    text=steps_df["avg_steps"].round(0).astype(int).astype(str),
    textposition="outside",
    hovertemplate="%{x}: %{customdata[0]}<br>Avg steps: %{y:.0f}<extra></extra>",
)
fig.add_hline(
    y=520,
    line_dash="dot",
    line_color="#898781",
    annotation_text="max_steps (520)",
    annotation_position="top left",
    annotation_font_size=10,
    annotation_font_color="#898781",
)
fig.update_layout(
    title="libero_10 / Long: average steps to finish, per task",
    xaxis_title="Task",
    yaxis_title="Average steps",
    yaxis_range=[0, 560],
    template="plotly_white",
    height=500,
    width=850,
    margin=dict(l=10, r=10, t=60, b=40),
)
fig.show()

steps_df[["task_short", "instruction", "avg_steps", "success_pct", "n_fail"]]

**Takeaway:** `T8` is not just the lowest-success task; it also uses the most steps on average. That supports the interpretation that the policy makes partial progress but struggles to finish the second moka-pot placement before the timeout.

## 4. Effort vs. reliability

Success rate alone hides how hard a task was even when it worked. The step-budget ratio (`n_steps / max_steps`, successes only) captures how much of the available horizon the policy used when it succeeded.

In [ ]:
succ = results[results["success"] == 1].copy()
step_ratio = (
    succ.assign(ratio=succ["n_steps"] / succ["max_steps"])
    .groupby("task_id")["ratio"]
    .mean()
    .rename("step_budget_ratio")
)
task = task.merge(step_ratio, on="task_id", how="left")
task["step_budget_pct"] = task["step_budget_ratio"] * 100

def declutter_1d(values, min_gap):
    order = sorted(range(len(values)), key=lambda i: values[i])
    adjusted = list(values)
    for k in range(1, len(order)):
        i, prev = order[k], order[k - 1]
        if adjusted[i] - adjusted[prev] < min_gap:
            adjusted[i] = adjusted[prev] + min_gap
    return adjusted

task = task.sort_values("task_num").reset_index(drop=True)
task["step_budget_x"] = declutter_1d(task["step_budget_pct"].tolist(), min_gap=1.8)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=task["step_budget_x"],
    y=task["success_pct"],
    mode="markers+text",
    text=task["task_short"],
    textposition="top center",
    textfont=dict(size=10),
    customdata=task[["instruction", "step_budget_pct"]],
    hovertemplate="%{text}: %{customdata[0]}<br>Step-budget=%{customdata[1]:.0f}%, Success=%{y:.0f}%<extra></extra>",
    marker=dict(size=14, color="#2a78d6"),
))
fig.add_hline(
    y=overall,
    line_dash="dot",
    line_color="#898781",
    annotation_text=f"suite avg success {overall:.1f}%",
    annotation_position="bottom right",
)
fig.add_vline(
    x=task["step_budget_pct"].median(),
    line_dash="dot",
    line_color="#898781",
    annotation_text="median effort",
    annotation_position="top left",
)
fig.update_layout(
    title="libero_10 / Long: effort vs. reliability",
    xaxis_title="Mean step-budget used on successful episodes (%)",
    yaxis_title="Success rate (%)",
    xaxis_range=[25, 90],
    yaxis_range=[75, 103],
    template="plotly_white",
    height=540,
    width=850,
    margin=dict(l=10, r=10, t=60, b=40),
)
fig.show()

task[["task_short", "instruction", "step_budget_pct", "success_pct", "n_fail"]].sort_values("success_pct")

**Takeaway:** `T8` is both low-reliability and high-effort. Some tasks are relatively slow but perfectly reliable, but `T8` is the only task combining many failures with high average step use.

## 5. Scene-property correlations and task-structure checks

For Long, `initial_distance` may be unavailable or set to `-1`. That makes distance correlations invalid here. We can still check available per-task properties such as distractor density and object count, but with only 10 tasks this is exploratory, not a strong statistical test.

In [ ]:
# Pick columns that exist in this particular export.
candidate_cols = []
for c in ["distractor_density", "n_distractors", "n_objects_sim", "total_objects", "target_object_count", "target_objects", "subgoal_count"]:
    if c in task.columns:
        candidate_cols.append(c)

corr_rows = []
for c in candidate_cols:
    x = pd.to_numeric(task[c], errors="coerce")
    valid = x.notna()
    # Drop known sentinel values.
    valid &= x != -1
    if valid.sum() >= 3 and x[valid].nunique() > 1:
        r, p = spearmanr(x[valid], task.loc[valid, "success_pct"])
        corr_rows.append({"property": c, "n_tasks": int(valid.sum()), "spearman_r": r, "p_value": p})
    else:
        corr_rows.append({"property": c, "n_tasks": int(valid.sum()), "spearman_r": np.nan, "p_value": np.nan})

corr_df = pd.DataFrame(corr_rows)
display(corr_df)

# Plot distractor density if available.
if "distractor_density" in task.columns:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=task["distractor_density"],
        y=task["success_pct"],
        mode="markers+text",
        text=task["task_short"],
        textposition="top center",
        textfont=dict(size=10),
        customdata=task[["instruction"]],
        marker=dict(size=14, color="#2a78d6"),
        hovertemplate="%{text}: %{customdata[0]}<br>Density=%{x:.3f}, Success=%{y:.0f}%<extra></extra>",
    ))
    fig.update_layout(
        title="libero_10 / Long: distractor density vs. success rate",
        xaxis_title="Distractor density",
        yaxis_title="Success rate (%)",
        yaxis_range=[75, 103],
        template="plotly_white",
        height=500,
        width=760,
        margin=dict(l=10, r=10, t=60, b=40),
    )
    fig.show()

show_cols = ["task_short", "instruction", "success_pct"]
for c in candidate_cols:
    show_cols.append(c)
task[show_cols].sort_values("success_pct")

**Takeaway:** available scene properties do not cleanly explain Long failures by themselves. The main signal comes from task structure and the manual failure review: later subgoal completion, second-object placement, spatial placement, and close/final-state completion.

## 6. Failure timing

If failed episodes all hit `max_steps`, then failures are timeouts: the policy keeps trying but never satisfies the task goal. That is different from a runtime crash or an early decisive wrong action.

In [ ]:
results["outcome"] = results["success"].map({1: "Success", 0: "Failure"})
failures = results[results["success"] == 0]
all_timeouts = bool((failures["n_steps"] == failures["max_steps"]).all()) if len(failures) else True
print(f"Did every failure run the full step budget? {all_timeouts}")
print(f"Failure step counts: {sorted(failures['n_steps'].unique().tolist()) if len(failures) else []}")

fig = px.histogram(
    results,
    x="n_steps",
    color="outcome",
    color_discrete_map={"Success": "#2a78d6", "Failure": "#fab219"},
    nbins=50,
    barmode="overlay",
    opacity=0.85,
    labels={"n_steps": "Steps to finish the episode"},
    title="libero_10 / Long: steps-to-finish, successes vs. failures",
)
fig.update_layout(template="plotly_white", height=430, width=850, margin=dict(l=10, r=10, t=60, b=40))
fig.show()

**Takeaway:** all 16 Long failures hit the 520-step limit. The failure mode is not a crash and not an immediate quit; the model usually makes partial progress and then fails to complete the final accepted state before time runs out.

## 7. Manual failed-subgoal analysis

The Long eval did **not** automatically log environment-level subgoal completion states. Therefore, the failed subgoals below come from post-hoc manual review of the saved failure videos and the annotation file. This is the key Long-specific analysis artifact.

In [ ]:
if annotations.empty:
    print("No all_failure_video_subgoal_annotations.csv found. Add it to DATA_DIR to enable this section.")
else:
    annotations["task_short"] = annotations["task_id"].apply(lambda t: "T" + str(t).split("_")[-1])

    display_cols = [
        "task_short", "episode_idx", "failure_type", "failure_stage",
        "subgoal_1_status", "subgoal_2_status", "manual_note"
    ]
    display(annotations[display_cols].sort_values(["task_short", "episode_idx"]))

    type_counts = (
        annotations.groupby("failure_type", as_index=False)
        .size()
        .rename(columns={"size": "n_failures"})
        .sort_values("n_failures")
    )

    fig = go.Figure()
    fig.add_bar(
        x=type_counts["n_failures"],
        y=type_counts["failure_type"],
        orientation="h",
        marker_color="#fab219",
        text=type_counts["n_failures"].astype(str),
        textposition="outside",
    )
    fig.update_layout(
        title="libero_10 / Long: manual failure types",
        xaxis_title="Number of failed episodes",
        yaxis_title="Failure type",
        template="plotly_white",
        height=430,
        width=850,
        margin=dict(l=10, r=10, t=60, b=40),
    )
    fig.show()

    task_type = (
        annotations.groupby(["task_short", "failure_type"], as_index=False)
        .size()
        .rename(columns={"size": "n_failures"})
        .sort_values(["task_short", "n_failures"], ascending=[True, False])
    )
    display(task_type)

**Takeaway:** the strongest manual signal is `T8`: every failure happens after the first moka-pot placement, during the second similar-object placement. `T6` is also a second-subgoal failure, but for relative spatial placement. `T3` and `T9` are close-action / final-state completion failures.

## 8. Qualitative failure gallery / examples

This section attempts to show representative frames if failure videos are available locally. In the committed repo, videos may intentionally be omitted to keep the repository lightweight; in that case, the notebook falls back to a readable table of the manual annotations.

In [ ]:
from pathlib import Path

# Common locations where failure videos may exist.
candidate_video_dirs = [
    DATA_DIR / "videos" / "failures",
    DATA_DIR / "failure_videos",
    DATA_DIR,
]

def find_video(filename):
    for d in candidate_video_dirs:
        p = d / filename
        if p.exists():
            return p
    return None

if annotations.empty:
    print("No annotation file available.")
else:
    examples = []
    # One representative per observed failure type.
    for failure_type, group in annotations.groupby("failure_type"):
        row = group.sort_values(["task_id", "episode_idx"]).iloc[0]
        examples.append(row)

    available = []
    missing = []
    for row in examples:
        p = find_video(row["video_file"])
        if p is None:
            missing.append(row)
        else:
            available.append((row, p))

    if not available:
        print("No local failure videos found. Showing the manual-review table instead.")
        display(annotations[["task_short", "episode_idx", "video_file", "failure_type", "failure_stage", "manual_note"]])
    else:
        import cv2
        import matplotlib.pyplot as plt

        def get_frame(video_path, fraction=0.99):
            cap = cv2.VideoCapture(str(video_path))
            n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            idx = min(int(n_frames * fraction), max(n_frames - 1, 0))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            cap.release()
            return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None

        n = len(available)
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5))
        if n == 1:
            axes = [axes]
        for ax, (row, video_path) in zip(axes, available):
            frame = get_frame(video_path, 0.99)
            if frame is not None:
                ax.imshow(frame)
            ax.axis("off")
            ax.set_title(f"{row['task_short']} ep{row['episode_idx']}\n{row['failure_type']}", fontsize=9)
        plt.tight_layout()
        plt.show()

        if missing:
            print(f"{len(missing)} representative videos were not found locally; see table below.")
            display(pd.DataFrame(missing)[["task_short", "episode_idx", "video_file", "failure_type", "failure_stage"]])

## 9. Task complexity: exploratory grouping

This section checks whether failures line up with instruction structure: multi-object tasks, open/close tasks, spatial-relation tasks, appliance tasks, and two-subgoal tasks. Because there are only 10 tasks, this is descriptive rather than a formal statistical claim.

In [ ]:
# Create simple instruction-derived flags if scene/subgoal columns are not already present.
if "has_multiple_objects" not in task.columns:
    task["has_multiple_objects"] = task["instruction"].str.contains("both| and ", case=False, regex=True).astype(int)
if "has_open_close" not in task.columns:
    task["has_open_close"] = task["instruction"].str.contains("close|open|drawer|microwave", case=False, regex=True).astype(int)
if "has_spatial_relation" not in task.columns:
    task["has_spatial_relation"] = task["instruction"].str.contains("left|right|back|bottom|inside|on top|to the right", case=False, regex=True).astype(int)
if "has_appliance_interaction" not in task.columns:
    task["has_appliance_interaction"] = task["instruction"].str.contains("stove|microwave", case=False, regex=True).astype(int)

flag_cols = ["has_multiple_objects", "has_open_close", "has_spatial_relation", "has_appliance_interaction"]
existing_flags = [c for c in flag_cols if c in task.columns]

summary_frames = []
for flag in existing_flags:
    tmp = (
        task.groupby(flag)
        .agg(
            n_tasks=("task_id", "count"),
            avg_success=("success_pct", "mean"),
            total_failures=("n_fail", "sum"),
            tasks=("task_short", lambda s: ", ".join(s)),
        )
        .reset_index()
    )
    tmp.insert(0, "flag", flag)
    tmp = tmp.rename(columns={flag: "flag_value"})
    summary_frames.append(tmp)

if summary_frames:
    flag_summary = pd.concat(summary_frames, ignore_index=True)
    flag_summary["avg_success"] = flag_summary["avg_success"].round(1)
    display(flag_summary)
else:
    print("No structure flags available.")

# Show failed task structures.
show_cols = ["task_short", "instruction", "success_pct", "n_fail"] + existing_flags
task[show_cols].sort_values("success_pct")

**Takeaway:** Long failures are best explained by task structure rather than by a single global scene property. The hardest cases require the policy to maintain state across subgoals and satisfy precise final conditions: second similar-object placement (`T8`), relative spatial placement (`T6`), and open/close final-state completion (`T3`, `T9`).

## Findings summary

- **Overall success rate: 96.8%** (484/500 episodes). Performance is strong overall, but this is the lowest suite-level success among the four suites.
- **6 of 10 tasks are perfect** with 100% success.
- **`T8` is the clear outlier**: "put both moka pots on the stove" reaches 41/50 = 82%, contributes 9/16 failures, and has the highest average step count.
- **Every failure is a timeout** at 520 steps. These are incomplete task-execution failures, not runtime crashes.
- **Failures are later-subgoal failures more than first-step failures.** Manual review shows that failed episodes usually make partial progress before stalling.
- **Main failure clusters:**
  - `T8`: second similar-object placement timeout after one moka pot is already placed.
  - `T6`: second-subgoal relative spatial placement failure after the mug-on-plate subgoal.
  - `T3`: drawer/cabinet close-action or accepted-closed-state failure after bowl placement.
  - `T9`: microwave close-action / final-state completion failure after the mug reaches the microwave.
- **Interpretation:** Long-suite residual errors come from sequencing, state maintenance, precision placement, and final-condition completion, not from broad instruction misunderstanding.

## Recommendations & mitigations

Given the project scope, we do not retrain MolmoAct2 here. The recommendations below are lightweight extensions or future-work directions tied directly to the observed Long-suite failures.

**Problem 1: second similar-object placement fails in `T8`**
- **Targeted fine-tuning:** add more examples where two visually similar objects must be placed at the same target location.
- **No-training patch:** after the first object is placed, explicitly re-detect which matching object remains unplaced and bias the next subgoal toward that remaining object.
- **Evaluation extension:** log subgoal-level completion events so the "first moka pot placed / second moka pot failed" pattern becomes automatic rather than manual.

**Problem 2: relative spatial placement fails in `T6`**
- **Targeted fine-tuning:** add examples where one object is placed relative to another already-placed object, especially "to the right of" relations.
- **No-training patch:** after the first subgoal, compute/estimate the reference object location and use a local crop or spatial check before the second placement.

**Problem 3: close/final-state completion fails in `T3` and `T9`**
- **Targeted fine-tuning:** add more container/appliance close-action examples where success depends on the final closed state.
- **No-training patch:** add a post-action verification loop: if the drawer/microwave is not closed, keep the policy focused on the close subgoal rather than drifting to other manipulation attempts.

**Most useful next notebook improvement:** add automatic event logging inside the eval script: first close target, grasp count, object displacement, subgoal state, and final object/receptacle distance. That would make Long directly comparable to the more instrumented Object/Spatial analysis.